In [ ]:
!pip install mne

In [ ]:
"""
BrainPrint - Phase 1 & 2
Load PhysioNet EEG Motor Movement/Imagery data, filter it, cut it into
windows and extract simple features (band power, mean, std).

Install:  pip install mne scipy numpy pandas matplotlib
Run:      python brainprint_features.py
Output:   features.csv  +  eeg_preview.png
"""
import numpy as np
import pandas as pd
import mne
from mne.datasets import eegbci
from scipy.signal import welch
import matplotlib.pyplot as plt

mne.set_log_level("WARNING")

In [ ]:
# ------------------------- SETTINGS (change here) -------------------------
SUBJECTS = list(range(1, 11))      # start with 10 people (1..109 available)
RUNS = [1, 2, 3, 4, 5, 6, 7, 8]    # each run = one separate recording
# run 1 = eyes open (rest), run 2 = eyes closed (rest), runs 3-8 = motor tasks
WINDOW_SEC = 4.0                   # length of each window
LOW_HZ, HIGH_HZ = 1.0, 40.0        # band-pass filter

BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 40),
}

In [ ]:
# ------------------------- PHASE 1: load + filter -------------------------
def load_run(subject, run):
    """Download (first time only) and load one recording, then filter it."""
    path = eegbci.load_data(subject, runs=[run])[0]
    raw = mne.io.read_raw_edf(path, preload=True)
    eegbci.standardize(raw)                      # clean channel names (Fc5. -> FC5)
    raw.filter(LOW_HZ, HIGH_HZ, fir_design="firwin")
    return raw


def plot_preview(raw, filename="eeg_preview.png", n_channels=3, seconds=10):
    """Save a quick plot of a few channels (good for your slides)."""
    sfreq = raw.info["sfreq"]
    data = raw.get_data()[:n_channels, : int(seconds * sfreq)] * 1e6  # volts -> microvolts
    t = np.arange(data.shape[1]) / sfreq
    fig, axes = plt.subplots(n_channels, 1, sharex=True, figsize=(10, 5))
    for i, ax in enumerate(axes):
        ax.plot(t, data[i], lw=0.8)
        ax.set_ylabel(f"{raw.ch_names[i]} (uV)")
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle("Filtered EEG (1-40 Hz)")
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)

In [ ]:
# ------------------------- PHASE 2: windows + features --------------------
def make_windows(raw):
    """Cut the recording into fixed-length windows -> array (n_windows, n_channels, n_samples)."""
    epochs = mne.make_fixed_length_epochs(
        raw, duration=WINDOW_SEC, preload=True, verbose=False
    )
    return epochs.get_data() * 1e6   # microvolts


def extract_features(windows, sfreq, ch_names):
    """
    For every window and every channel compute:
      - log absolute power in each band (delta..gamma)
      - relative power in each band (band / total power)
      - mean and standard deviation of the signal
    Returns (feature_matrix, feature_names).
    """
    freqs, psd = welch(windows, fs=sfreq, nperseg=int(sfreq * 2), axis=-1)
    df = freqs[1] - freqs[0]
    total_mask = (freqs >= LOW_HZ) & (freqs <= HIGH_HZ)
    total = psd[..., total_mask].sum(-1) * df                # (n, ch)

    feats, names = [], []
    for band, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs < hi)
        power = psd[..., mask].sum(-1) * df                  # (n, ch)
        feats.append(np.log10(power + 1e-12))
        names += [f"{ch}_{band}_logpow" for ch in ch_names]
        feats.append(power / (total + 1e-12))
        names += [f"{ch}_{band}_relpow" for ch in ch_names]

    feats.append(windows.mean(-1))
    names += [f"{ch}_mean" for ch in ch_names]
    feats.append(windows.std(-1))
    names += [f"{ch}_std" for ch in ch_names]

    return np.concatenate(feats, axis=1), names


In [ ]:
# ------------------------- MAIN -------------------------------------------
def main():
    tables = []
    for subject in SUBJECTS:
        for run in RUNS:
            print(f"Subject {subject:3d} | run {run:2d}")
            raw = load_run(subject, run)

            if subject == SUBJECTS[0] and run == RUNS[0]:
                plot_preview(raw)

            windows = make_windows(raw)
            X, names = extract_features(windows, raw.info["sfreq"], raw.ch_names)

            df = pd.DataFrame(X, columns=names)
            df.insert(0, "window", np.arange(len(df)))
            df.insert(0, "run", run)         # needed to split BY RECORDING later
            df.insert(0, "subject", subject) # the label (identity) to predict
            tables.append(df)

    features = pd.concat(tables, ignore_index=True)
    features.to_csv("features.csv", index=False)
    print(f"\nSaved features.csv: {features.shape[0]} windows x {features.shape[1]} columns")
    print(features.groupby("subject").size())


if __name__ == "__main__":
    main()

Subject   1 | run  1


Do you want to set the path:
    /root/mne_data
as the default EEGBCI dataset path in the mne-python config [y]/n? y


Subject   1 | run  2


Subject   1 | run  3


Subject   1 | run  4
Subject   1 | run  5


Subject   1 | run  6


... (59 outputs left out)

Subject  10 | run  5


Subject  10 | run  6


Subject  10 | run  7


Subject  10 | run  8

Saved features.csv: 2118 windows x 771 columns
subject
1     216
2     210
3     216
4     210
5     210
6     210
7     216
8     210
9     210
10    210
dtype: int64


In [ ]:
"""
BrainPrint - Phase 3
Train Random Forest + baselines, with the split done BY RECORDING (run).

Input:   features.csv  (from brainprint_features.py)
Output:  confusion_matrix.png, feature_importance.png, brainprint_rf.joblib
"""
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# ------------------------- SETTINGS -------------------------
TRAIN_RUNS = [1, 2, 3, 4, 5, 6]   # recordings used for training
TEST_RUNS = [7, 8]                # recordings NEVER seen during training
SEED = 42

df = pd.read_csv("features.csv")
meta_cols = ["subject", "run", "window"]
feature_cols = [c for c in df.columns if c not in meta_cols]

train = df[df["run"].isin(TRAIN_RUNS)]
test = df[df["run"].isin(TEST_RUNS)]
X_train, y_train = train[feature_cols], train["subject"]
X_test, y_test = test[feature_cols], test["subject"]
n_subjects = df["subject"].nunique()

print(f"Train: {len(train)} windows (runs {TRAIN_RUNS})")
print(f"Test : {len(test)} windows (runs {TEST_RUNS})")
print(f"Random-guess level: {100 / n_subjects:.1f}%\n")

# ------------------------- MODELS -------------------------
models = {
    "Baseline (random guess)": DummyClassifier(strategy="stratified", random_state=SEED),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10)),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = acc
    print(f"{name:26s} accuracy = {acc * 100:5.1f}%")

# ------------------------- CONFUSION MATRIX (Random Forest) -------------------------
rf = models["Random Forest"]
pred = rf.predict(X_test)
cm = confusion_matrix(y_test, pred, labels=sorted(df["subject"].unique()))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=sorted(df["subject"].unique())).plot(
    ax=ax, cmap="Blues", colorbar=False
)
ax.set_title(f"Random Forest - test on unseen runs {TEST_RUNS}")
ax.set_xlabel("Predicted subject")
ax.set_ylabel("True subject")
fig.tight_layout()
fig.savefig("confusion_matrix.png", dpi=150)
plt.close(fig)

# ------------------------- TOP FEATURES -------------------------
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 10 features:")
print(imp.head(10).round(4).to_string())
fig, ax = plt.subplots(figsize=(8, 4))
imp.head(15)[::-1].plot.barh(ax=ax)
ax.set_title("Top 15 most important features (Random Forest)")
fig.tight_layout()
fig.savefig("feature_importance.png", dpi=150)
plt.close(fig)

# ------------------------- ROBUSTNESS: leave-one-run-out -------------------------
# Each recording takes a turn as the test set (the other runs train the model).
print("\nLeave-one-run-out (Random Forest):")
loro = []
for run in sorted(df["run"].unique()):
    tr, te = df[df["run"] != run], df[df["run"] == run]
    m = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    m.fit(tr[feature_cols], tr["subject"])
    acc = accuracy_score(te["subject"], m.predict(te[feature_cols]))
    loro.append(acc)
    print(f"  test on run {run}: {acc * 100:5.1f}%")
print(f"  mean = {np.mean(loro) * 100:.1f}%  (std {np.std(loro) * 100:.1f}%)")

# ------------------------- SAVE MODEL FOR STREAMLIT -------------------------
joblib.dump({"model": rf, "features": feature_cols}, "brainprint_rf.joblib")
print("\nSaved: confusion_matrix.png, feature_importance.png, brainprint_rf.joblib")

Train: 1512 windows (runs [1, 2, 3, 4, 5, 6])
Test : 606 windows (runs [7, 8])
Random-guess level: 10.0%

Baseline (random guess)    accuracy =  12.0%
Logistic Regression        accuracy =  99.8%
SVM (RBF)                  accuracy =  99.3%
Random Forest              accuracy =  99.0%

Top 10 features:
T10_beta_logpow     0.0215
T10_gamma_logpow    0.0205
T10_std             0.0188
T7_beta_logpow      0.0158
T7_gamma_logpow     0.0143
T10_theta_logpow    0.0134
TP7_std             0.0103
FT8_alpha_logpow    0.0103
TP7_theta_logpow    0.0097
F8_beta_logpow      0.0095

Leave-one-run-out (Random Forest):
  test on run 1:  98.7%
  test on run 2:  74.7%
  test on run 3:  99.7%
  test on run 4: 100.0%
  test on run 5:  98.3%
  test on run 6:  99.3%
  test on run 7:  99.7%
  test on run 8:  99.7%
  mean = 96.3%  (std 8.2%)

Saved: confusion_matrix.png, feature_importance.png, brainprint_rf.joblib


In [ ]:
"""
BrainPrint - Phase 4: experiments to understand and improve identification
  A. Ablation: which features really carry identity?
  B. Accuracy vs number of subjects
  C. Accuracy vs number of channels

Input:   features.csv
Output:  ablation_results.csv, ablation.png, acc_vs_subjects.png, acc_vs_channels.png
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

TRAIN_RUNS = [1, 2, 3, 4, 5, 6]
TEST_RUNS = [7, 8]
SEED = 42
REPEATS = 5          # random repetitions for experiments B and C
N_TREES = 200

df = pd.read_csv("features.csv")
meta_cols = ["subject", "run", "window"]
feature_cols = [c for c in df.columns if c not in meta_cols]


def channel_of(col):
    """'FC5_alpha_logpow' -> 'FC5' ; 'T10_std' -> 'T10'"""
    if col.endswith("_logpow") or col.endswith("_relpow"):
        return col.rsplit("_", 2)[0]
    return col.rsplit("_", 1)[0]


def band_of(col):
    if col.endswith("_logpow") or col.endswith("_relpow"):
        return col.rsplit("_", 2)[1]
    return None


def new_rf():
    return RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1)


def split_accuracy(data, cols):
    tr, te = data[data["run"].isin(TRAIN_RUNS)], data[data["run"].isin(TEST_RUNS)]
    m = new_rf().fit(tr[cols], tr["subject"])
    return accuracy_score(te["subject"], m.predict(te[cols]))


def loro_accuracy(data, cols):
    """Leave-one-run-out: each recording takes a turn as test set."""
    accs = []
    for run in sorted(data["run"].unique()):
        tr, te = data[data["run"] != run], data[data["run"] == run]
        m = new_rf().fit(tr[cols], tr["subject"])
        accs.append(accuracy_score(te["subject"], m.predict(te[cols])))
    return np.mean(accs), np.std(accs)


# ======================= A. ABLATION =======================
TEMPORAL = {"T7", "T8", "T9", "T10", "TP7", "TP8", "FT7", "FT8"}

feature_sets = {
    "All features": feature_cols,
    "No gamma": [c for c in feature_cols if band_of(c) != "gamma"],
    "No temporal channels": [c for c in feature_cols if channel_of(c) not in TEMPORAL],
    "No gamma + no temporal": [
        c for c in feature_cols
        if band_of(c) != "gamma" and channel_of(c) not in TEMPORAL
    ],
    "Only theta + alpha": [c for c in feature_cols if band_of(c) in ("theta", "alpha")],
    "Only relative power": [c for c in feature_cols if c.endswith("_relpow")],
    "Only log power": [c for c in feature_cols if c.endswith("_logpow")],
    "Only mean + std": [c for c in feature_cols if c.endswith("_mean") or c.endswith("_std")],
}

print("=== A. Ablation (Random Forest) ===")
rows = []
for name, cols in feature_sets.items():
    acc_split = split_accuracy(df, cols)
    acc_loro, std_loro = loro_accuracy(df, cols)
    rows.append({"feature_set": name, "n_features": len(cols),
                 "acc_test_runs_7_8": acc_split,
                 "acc_loro_mean": acc_loro, "acc_loro_std": std_loro})
    print(f"{name:26s} n={len(cols):4d} | test 7-8: {acc_split*100:5.1f}% "
          f"| LORO: {acc_loro*100:5.1f}% (+-{std_loro*100:.1f})")

res = pd.DataFrame(rows)
res.to_csv("ablation_results.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(res["feature_set"][::-1], res["acc_loro_mean"][::-1] * 100,
        xerr=res["acc_loro_std"][::-1] * 100, color="#4C78A8")
ax.axvline(100 / df["subject"].nunique(), color="red", ls="--", label="random guess")
ax.set_xlabel("Leave-one-run-out accuracy (%)")
ax.set_title("Ablation: which features carry identity?")
ax.legend()
fig.tight_layout()
fig.savefig("ablation.png", dpi=150)
plt.close(fig)

# ======================= B. ACCURACY vs NUMBER OF SUBJECTS =======================
print("\n=== B. Accuracy vs number of subjects ===")
rng = np.random.default_rng(SEED)
all_subjects = sorted(df["subject"].unique())
sizes = [n for n in [2, 3, 4, 5, 6, 8, 10, 15, 20] if n <= len(all_subjects)]
mean_b, std_b = [], []
for n in sizes:
    accs = []
    for _ in range(REPEATS):
        chosen = rng.choice(all_subjects, size=n, replace=False)
        accs.append(split_accuracy(df[df["subject"].isin(chosen)], feature_cols))
    mean_b.append(np.mean(accs)); std_b.append(np.std(accs))
    print(f"{n:3d} subjects: {np.mean(accs)*100:5.1f}% (+-{np.std(accs)*100:.1f})")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(sizes, np.array(mean_b) * 100, yerr=np.array(std_b) * 100, marker="o", label="Random Forest")
ax.plot(sizes, [100 / n for n in sizes], "r--", label="random guess")
ax.set_xlabel("Number of subjects")
ax.set_ylabel("Accuracy on unseen runs (%)")
ax.set_title("Accuracy vs number of subjects")
ax.set_ylim(0, 105)
ax.legend()
fig.tight_layout()
fig.savefig("acc_vs_subjects.png", dpi=150)
plt.close(fig)

# ======================= C. ACCURACY vs NUMBER OF CHANNELS =======================
print("\n=== C. Accuracy vs number of channels ===")
channels = sorted({channel_of(c) for c in feature_cols})
ch_sizes = [n for n in [1, 2, 4, 8, 16, 32, 64] if n <= len(channels)]
mean_c, std_c = [], []
for n in ch_sizes:
    accs = []
    for _ in range(REPEATS):
        chosen = set(rng.choice(channels, size=n, replace=False))
        cols = [c for c in feature_cols if channel_of(c) in chosen]
        accs.append(split_accuracy(df, cols))
    mean_c.append(np.mean(accs)); std_c.append(np.std(accs))
    print(f"{n:3d} channels: {np.mean(accs)*100:5.1f}% (+-{np.std(accs)*100:.1f})")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(ch_sizes, np.array(mean_c) * 100, yerr=np.array(std_c) * 100, marker="o")
ax.axhline(100 / df["subject"].nunique(), color="red", ls="--", label="random guess")
ax.set_xscale("log", base=2)
ax.set_xlabel("Number of channels (random subsets)")
ax.set_ylabel("Accuracy on unseen runs (%)")
ax.set_title("Accuracy vs number of channels")
ax.set_ylim(0, 105)
ax.legend()
fig.tight_layout()
fig.savefig("acc_vs_channels.png", dpi=150)
plt.close(fig)

print("\nSaved: ablation_results.csv, ablation.png, acc_vs_subjects.png, acc_vs_channels.png")

=== A. Ablation (Random Forest) ===
All features               n= 768 | test 7-8:  99.0% | LORO:  96.1% (+-8.1)
No gamma                   n= 640 | test 7-8:  99.0% | LORO:  96.2% (+-7.7)
No temporal channels       n= 672 | test 7-8:  97.9% | LORO:  95.4% (+-8.9)
No gamma + no temporal     n= 560 | test 7-8:  98.0% | LORO:  93.7% (+-10.4)
Only theta + alpha         n= 256 | test 7-8:  95.9% | LORO:  92.4% (+-10.1)
Only relative power        n= 320 | test 7-8:  94.2% | LORO:  89.2% (+-12.7)
Only log power             n= 320 | test 7-8:  99.0% | LORO:  96.5% (+-7.8)
Only mean + std            n= 128 | test 7-8:  97.2% | LORO:  93.2% (+-7.5)

=== B. Accuracy vs number of subjects ===
  2 subjects: 100.0% (+-0.0)
  3 subjects:  99.8% (+-0.4)
  4 subjects:  99.1% (+-0.8)
  5 subjects:  99.5% (+-0.6)
  6 subjects:  99.4% (+-0.7)
  8 subjects:  99.0% (+-0.1)
 10 subjects:  99.0% (+-0.0)

=== C. Accuracy vs number of channels ===
  1 channels:  76.5% (+-1.4)
  2 channels:  82.0% (+-3.2)
  4 ch

In [ ]:
%%writefile brainprint_features.py
import numpy as np
import pandas as pd
import mne
from mne.datasets import eegbci
from scipy.signal import welch
import matplotlib.pyplot as plt

mne.set_log_level("WARNING")

SUBJECTS = list(range(1, 11))
RUNS = [1, 2, 3, 4, 5, 6, 7, 8]
WINDOW_SEC = 4.0
LOW_HZ, HIGH_HZ = 1.0, 40.0

BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 40),
}


def load_run(subject, run):
    path = eegbci.load_data(subject, runs=[run])[0]
    raw = mne.io.read_raw_edf(path, preload=True)
    eegbci.standardize(raw)
    raw.filter(LOW_HZ, HIGH_HZ, fir_design="firwin")
    return raw


def extract_features(windows, sfreq, ch_names):
    freqs, psd = welch(windows, fs=sfreq, nperseg=int(sfreq * 2), axis=-1)
    df = freqs[1] - freqs[0]
    total_mask = (freqs >= LOW_HZ) & (freqs <= HIGH_HZ)
    total = psd[..., total_mask].sum(-1) * df

    feats, names = [], []
    for band, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs < hi)
        power = psd[..., mask].sum(-1) * df
        feats.append(np.log10(power + 1e-12))
        names += [f"{ch}_{band}_logpow" for ch in ch_names]
        feats.append(power / (total + 1e-12))
        names += [f"{ch}_{band}_relpow" for ch in ch_names]

    feats.append(windows.mean(-1))
    names += [f"{ch}_mean" for ch in ch_names]
    feats.append(windows.std(-1))
    names += [f"{ch}_std" for ch in ch_names]

    return np.concatenate(feats, axis=1), names

Writing brainprint_features.py


In [ ]:
"""
BrainPrint - Phase 5: improve and stress-test identification
  1. Extend to 20 subjects (downloads subjects 11-20 the first time)
  2. Window length: 2 s vs 4 s vs 8 s
  3. Accuracy vs number of subjects (up to 20)
  4. State-shift test: train on tasks, test on rest (eyes open / eyes closed), and reverse
  5. Extra features: theta/beta ratio

Needs brainprint_features.py in the same folder (it is imported, not modified).
Output: features_<w>s_<n>subj.csv, phase5_results.csv,
        window_length.png, acc_vs_subjects_20.png, state_shift.png
"""
import os
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import brainprint_features as bf

mne.set_log_level("WARNING")

# ------------------------- SETTINGS -------------------------
SUBJECTS = list(range(1, 21))
RUNS = [1, 2, 3, 4, 5, 6, 7, 8]
WINDOWS = [2.0, 4.0, 8.0]
TRAIN_RUNS = [1, 2, 3, 4, 5, 6]
TEST_RUNS = [7, 8]
SEED = 42
N_TREES = 200
REPEATS = 3
META = ["subject", "run", "window"]


# ------------------------- BUILD / LOAD FEATURES -------------------------
def build_features():
    files = {w: f"features_{int(w)}s_{len(SUBJECTS)}subj.csv" for w in WINDOWS}
    if all(os.path.exists(f) for f in files.values()):
        print("Found saved feature files, loading them.")
        return {w: pd.read_csv(f) for w, f in files.items()}

    tables = {w: [] for w in WINDOWS}
    for s in SUBJECTS:
        for r in RUNS:
            print(f"Subject {s:3d} | run {r:2d}")
            raw = bf.load_run(s, r)           # filtered once, reused for all window lengths
            for w in WINDOWS:
                epochs = mne.make_fixed_length_epochs(raw, duration=w, preload=True, verbose=False)
                X, names = bf.extract_features(epochs.get_data() * 1e6, raw.info["sfreq"], raw.ch_names)
                df = pd.DataFrame(X, columns=names)
                df.insert(0, "window", np.arange(len(df)))
                df.insert(0, "run", r)
                df.insert(0, "subject", s)
                tables[w].append(df)

    out = {}
    for w in WINDOWS:
        out[w] = pd.concat(tables[w], ignore_index=True)
        out[w].to_csv(files[w], index=False)
    return out


# ------------------------- HELPERS -------------------------
def new_rf():
    return RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1)


def feature_columns(df):
    return [c for c in df.columns if c not in META]


def acc_train_test(df, cols, train_runs, test_runs):
    tr, te = df[df["run"].isin(train_runs)], df[df["run"].isin(test_runs)]
    m = new_rf().fit(tr[cols], tr["subject"])
    return accuracy_score(te["subject"], m.predict(te[cols]))


def loro(df, cols):
    accs = []
    for run in sorted(df["run"].unique()):
        accs.append(acc_train_test(df, cols, [r for r in df["run"].unique() if r != run], [run]))
    return np.mean(accs), np.std(accs)


def add_ratio(df):
    """theta/beta ratio per channel (difference of log powers = log of the ratio)."""
    new = {}
    for c in df.columns:
        if c.endswith("_theta_logpow"):
            ch = c[: -len("_theta_logpow")]
            new[f"{ch}_theta_beta_ratio"] = df[c] - df[f"{ch}_beta_logpow"]
    return pd.concat([df, pd.DataFrame(new)], axis=1)


# ------------------------- RUN -------------------------
data = build_features()
rows = []

# ===== 1. Window length =====
print("\n=== Window length (20 subjects) ===")
loro_by_w = {}
for w in WINDOWS:
    df = data[w]
    cols = feature_columns(df)
    a_split = acc_train_test(df, cols, TRAIN_RUNS, TEST_RUNS)
    a_loro, s_loro = loro(df, cols)
    loro_by_w[w] = (a_loro, s_loro)
    rows.append({"experiment": "window_length", "setting": f"{int(w)}s",
                 "test_7_8": a_split, "loro_mean": a_loro, "loro_std": s_loro})
    print(f"{int(w)} s windows ({len(df)} windows): test 7-8 {a_split*100:5.1f}% | "
          f"LORO {a_loro*100:5.1f}% (+-{s_loro*100:.1f})")

best_w = max(loro_by_w, key=lambda k: loro_by_w[k][0])
print(f"--> Best window length: {int(best_w)} s (used for the next experiments)")
df = data[best_w]
cols = feature_columns(df)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([f"{int(w)} s" for w in WINDOWS], [loro_by_w[w][0] * 100 for w in WINDOWS],
       yerr=[loro_by_w[w][1] * 100 for w in WINDOWS], color="#4C78A8")
ax.set_ylabel("Leave-one-run-out accuracy (%)")
ax.set_title("Effect of window length (20 subjects)")
ax.set_ylim(0, 105)
fig.tight_layout()
fig.savefig("window_length.png", dpi=150)
plt.close(fig)

# ===== 2. Accuracy vs number of subjects =====
print("\n=== Accuracy vs number of subjects ===")
rng = np.random.default_rng(SEED)
all_subj = sorted(df["subject"].unique())
sizes = [n for n in [2, 5, 8, 10, 12, 15, 18, 20] if n <= len(all_subj)]
means, stds = [], []
for n in sizes:
    accs = []
    for _ in range(REPEATS):
        chosen = rng.choice(all_subj, size=n, replace=False)
        accs.append(acc_train_test(df[df["subject"].isin(chosen)], cols, TRAIN_RUNS, TEST_RUNS))
    means.append(np.mean(accs)); stds.append(np.std(accs))
    rows.append({"experiment": "n_subjects", "setting": n, "test_7_8": np.mean(accs),
                 "loro_mean": np.nan, "loro_std": np.std(accs)})
    print(f"{n:3d} subjects: {np.mean(accs)*100:5.1f}% (+-{np.std(accs)*100:.1f})")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(sizes, np.array(means) * 100, yerr=np.array(stds) * 100, marker="o", label="Random Forest")
ax.plot(sizes, [100 / n for n in sizes], "r--", label="random guess")
ax.set_xlabel("Number of subjects")
ax.set_ylabel("Accuracy on unseen runs (%)")
ax.set_title("Accuracy vs number of subjects")
ax.set_ylim(0, 105)
ax.legend()
fig.tight_layout()
fig.savefig("acc_vs_subjects_20.png", dpi=150)
plt.close(fig)

# ===== 3. State-shift test =====
print("\n=== State-shift test ===")
TASKS = [3, 4, 5, 6, 7, 8]
shift_tests = {
    "Train tasks -> test eyes open (run 1)": (TASKS, [1]),
    "Train tasks -> test eyes closed (run 2)": (TASKS, [2]),
    "Train tasks -> test tasks (run 8)": ([3, 4, 5, 6, 7], [8]),
    "Train rest (1,2) -> test tasks (3-8)": ([1, 2], TASKS),
}
shift_res = {}
for name, (tr_runs, te_runs) in shift_tests.items():
    a = acc_train_test(df, cols, tr_runs, te_runs)
    shift_res[name] = a
    rows.append({"experiment": "state_shift", "setting": name, "test_7_8": a,
                 "loro_mean": np.nan, "loro_std": np.nan})
    print(f"{name:42s} {a*100:5.1f}%")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(shift_res.keys())[::-1], [v * 100 for v in list(shift_res.values())[::-1]], color="#F58518")
ax.axvline(100 / len(all_subj), color="red", ls="--", label="random guess")
ax.set_xlabel("Accuracy (%)")
ax.set_title("State-shift test: does identity survive a change of mental state?")
ax.legend()
fig.tight_layout()
fig.savefig("state_shift.png", dpi=150)
plt.close(fig)

# ===== 4. Extra features: theta/beta ratio =====
print("\n=== Extra features: theta/beta ratio ===")
df_r = add_ratio(df)
cols_r = feature_columns(df_r)
for label, d, c in [("Base features", df, cols), ("Base + theta/beta ratio", df_r, cols_r)]:
    a_loro, s_loro = loro(d, c)
    rows.append({"experiment": "extra_features", "setting": label, "test_7_8": np.nan,
                 "loro_mean": a_loro, "loro_std": s_loro})
    print(f"{label:26s} LORO {a_loro*100:5.1f}% (+-{s_loro*100:.1f})")

pd.DataFrame(rows).to_csv("phase5_results.csv", index=False)
print("\nSaved: phase5_results.csv, window_length.png, acc_vs_subjects_20.png, state_shift.png")

Subject   1 | run  1
Subject   1 | run  2
Subject   1 | run  3
Subject   1 | run  4
Subject   1 | run  5
Subject   1 | run  6
Subject   1 | run  7
Subject   1 | run  8
Subject   2 | run  1
Subject   2 | run  2
Subject   2 | run  3
Subject   2 | run  4
Subject   2 | run  5
Subject   2 | run  6
Subject   2 | run  7
Subject   2 | run  8
Subject   3 | run  1
Subject   3 | run  2
Subject   3 | run  3
Subject   3 | run  4
Subject   3 | run  5
Subject   3 | run  6
Subject   3 | run  7
Subject   3 | run  8
Subject   4 | run  1
Subject   4 | run  2
Subject   4 | run  3
Subject   4 | run  4
Subject   4 | run  5
Subject   4 | run  6
Subject   4 | run  7
Subject   4 | run  8
Subject   5 | run  1
Subject   5 | run  2
Subject   5 | run  3
Subject   5 | run  4
Subject   5 | run  5
Subject   5 | run  6
Subject   5 | run  7
Subject   5 | run  8
Subject   6 | run  1
Subject   6 | run  2
Subject   6 | run  3
Subject   6 | run  4
Subject   6 | run  5
Subject   6 | run  6
Subject   6 | run  7
Subject   6 |

Subject  11 | run  2


Subject  11 | run  3
Subject  11 | run  4


Subject  11 | run  5


Subject  11 | run  6


Subject  11 | run  7


... (58 outputs left out)

Subject  20 | run  5


Subject  20 | run  6


Subject  20 | run  7


Subject  20 | run  8

=== Window length (20 subjects) ===
2 s windows (8538 windows): test 7-8  97.0% | LORO  91.2% (+-14.4)
4 s windows (4218 windows): test 7-8  97.8% | LORO  92.2% (+-12.7)
8 s windows (2080 windows): test 7-8  98.0% | LORO  92.3% (+-13.3)
--> Best window length: 8 s (used for the next experiments)

=== Accuracy vs number of subjects ===
  2 subjects: 100.0% (+-0.0)
  5 subjects:  99.6% (+-0.3)
  8 subjects:  97.1% (+-1.5)
 10 subjects:  98.2% (+-1.8)
 12 subjects:  96.9% (+-1.5)
 15 subjects:  97.7% (+-0.4)
 18 subjects:  97.7% (+-0.9)
 20 subjects:  98.0% (+-0.0)

=== State-shift test ===
Train tasks -> test eyes open (run 1)       83.6%
Train tasks -> test eyes closed (run 2)     57.1%
Train tasks -> test tasks (run 8)           99.7%
Train rest (1,2) -> test tasks (3-8)        73.6%

=== Extra features: theta/beta ratio ===


In [6]:
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT = "/content/drive/MyDrive/BrainPrint"
os.chdir(PROJECT)
print(os.listdir(PROJECT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['mne_data', 'eeg_preview.png', 'features.csv', 'confusion_matrix.png', 'feature_importance.png', 'brainprint_rf.joblib', 'ablation_results.csv', 'ablation.png', 'acc_vs_subjects.png', 'acc_vs_channels.png', 'brainprint_features.py', '__pycache__', 'features_4s_30subj.csv', 'window_length.png', 'acc_vs_subjects_20.png', 'state_shift.png', 'phase5_results.csv']


In [7]:
"""
BrainPrint - Final model + authentication experiment

Part 1  Identification: train on all 30 subjects (runs 1-6), test on runs 7-8,
        save the model for the app.
Part 2  Authentication (verification): 20 subjects enrolled, 10 strangers.
        A person CLAIMS an identity; we accept if the model's probability for
        that identity >= threshold. Measures FAR / FRR / EER.

Input:   features_4s_30subj.csv  (from Phase 5)
Output:  brainprint_rf_30.joblib, brainprint_auth_rf.joblib,
         confusion_matrix_30.png, auth_far_frr.png, auth_curves.csv
"""
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# ------------------------- SETTINGS -------------------------
FEATURES_CSV = "features_4s_30subj.csv"
TRAIN_RUNS = [1, 2, 3, 4, 5, 6]
TEST_RUNS = [7, 8]
N_ENROLLED = 20          # the rest are "strangers"
N_SPLITS = 5             # random enrolled/stranger splits (to avoid a lucky split)
SEED = 42
N_TREES = 300
THR = np.linspace(0, 1, 101)

df = pd.read_csv(FEATURES_CSV)
META = ["subject", "run", "window"]
cols = [c for c in df.columns if c not in META]
all_subjects = sorted(df["subject"].unique())


def new_rf():
    return RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1)


def by_recording(sub_df, P):
    """Average the probabilities of all windows of the same recording (subject, run)."""
    tmp = pd.DataFrame(P)
    tmp["subject"] = sub_df["subject"].values
    tmp["run"] = sub_df["run"].values
    g = tmp.groupby(["subject", "run"]).mean().reset_index()
    return g.drop(columns=["subject", "run"]).values, g["subject"].values


# =====================================================================
# PART 1 - FINAL IDENTIFICATION MODEL (30 subjects)
# =====================================================================
print("=== PART 1: identification, %d subjects ===" % len(all_subjects))
tr = df[df["run"].isin(TRAIN_RUNS)]
te = df[df["run"].isin(TEST_RUNS)]
rf = new_rf().fit(tr[cols], tr["subject"])
P = rf.predict_proba(te[cols])
pred = rf.classes_[P.argmax(1)]
acc_win = accuracy_score(te["subject"], pred)

P_rec, subj_rec = by_recording(te, P)
acc_rec = accuracy_score(subj_rec, rf.classes_[P_rec.argmax(1)])
print(f"Window level    : {acc_win*100:5.1f}%  ({len(te)} windows, never seen in training)")
print(f"Recording level : {acc_rec*100:5.1f}%  ({len(subj_rec)} recordings, probabilities averaged)")
print(f"Random guess    : {100/len(all_subjects):5.1f}%")

cm = confusion_matrix(te["subject"], pred, labels=rf.classes_)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=rf.classes_).plot(
    ax=ax, cmap="Blues", colorbar=False, include_values=False)
ax.set_title(f"Identification, {len(all_subjects)} subjects (runs {TEST_RUNS}, unseen)")
ax.tick_params(labelsize=6)
fig.tight_layout()
fig.savefig("confusion_matrix_30.png", dpi=150)
plt.close(fig)

joblib.dump({"model": rf, "features": cols, "classes": list(rf.classes_),
             "window_sec": 4.0, "train_runs": TRAIN_RUNS}, "brainprint_rf_30.joblib")
print("Saved: brainprint_rf_30.joblib, confusion_matrix_30.png")


# =====================================================================
# PART 2 - AUTHENTICATION (VERIFICATION)
# =====================================================================
def attempts(P_e, subj_e, P_s, classes):
    """
    Build the three groups of scores. score = probability given to the CLAIMED identity.
      genuine      : enrolled person claims their own identity
      imp_known    : enrolled person claims someone else's identity
      imp_unknown  : stranger (never enrolled) claims any enrolled identity
    """
    idx = np.array([classes.index(s) for s in subj_e])
    rows = np.arange(len(idx))
    genuine = P_e[rows, idx]
    mask = np.ones_like(P_e, dtype=bool)
    mask[rows, idx] = False
    return genuine, P_e[mask], P_s.ravel()


def curves(genuine, imp_known, imp_unknown):
    imp_all = np.concatenate([imp_known, imp_unknown])
    frr = np.array([(genuine < t).mean() for t in THR])          # real user rejected
    far_all = np.array([(imp_all >= t).mean() for t in THR])     # impostor accepted
    far_unk = np.array([(imp_unknown >= t).mean() for t in THR]) # stranger accepted
    return frr, far_all, far_unk


def eer(frr, far):
    i = int(np.argmin(np.abs(far - frr)))
    return (far[i] + frr[i]) / 2, THR[i]


print("\n=== PART 2: authentication, %d enrolled / %d strangers, %d random splits ==="
      % (N_ENROLLED, len(all_subjects) - N_ENROLLED, N_SPLITS))

store = {"window": [], "recording": []}
eers = {"window": [], "recording": []}
first_split = None

for k in range(N_SPLITS):
    rng = np.random.default_rng(SEED + k)
    perm = list(rng.permutation(all_subjects))
    enrolled, strangers = sorted(perm[:N_ENROLLED]), sorted(perm[N_ENROLLED:])

    tr = df[df["subject"].isin(enrolled) & df["run"].isin(TRAIN_RUNS)]
    te = df[df["run"].isin(TEST_RUNS)]
    m = new_rf().fit(tr[cols], tr["subject"])          # the model only knows enrolled people
    classes = list(m.classes_)
    te_e, te_s = te[te["subject"].isin(enrolled)], te[te["subject"].isin(strangers)]
    P_e, P_s = m.predict_proba(te_e[cols]), m.predict_proba(te_s[cols])

    groups = {"window": attempts(P_e, te_e["subject"].values, P_s, classes)}
    Pe_r, sub_e = by_recording(te_e, P_e)
    Ps_r, _ = by_recording(te_s, P_s)
    groups["recording"] = attempts(Pe_r, sub_e, Ps_r, classes)

    for level, (g, ik, iu) in groups.items():
        frr, far_all, far_unk = curves(g, ik, iu)
        store[level].append((frr, far_all, far_unk))
        e, t = eer(frr, far_all)
        eers[level].append(e)
        if k == 0:
            first_split = first_split or {}
            first_split[level] = t
    if k == 0:
        first_model = {"model": m, "features": cols, "classes": classes,
                       "enrolled": enrolled, "strangers": strangers}
    print(f"split {k+1}/{N_SPLITS} done | EER window {eers['window'][-1]*100:.1f}% "
          f"| EER recording {eers['recording'][-1]*100:.1f}%")

# ---- summary
print("\nEqual Error Rate (EER, lower = better), mean +- std over splits:")
for level in ("window", "recording"):
    print(f"  {level:9s}: {np.mean(eers[level])*100:5.1f}% +- {np.std(eers[level])*100:.1f}")

curve_rows = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, level in zip(axes, ("window", "recording")):
    frr = np.mean([s[0] for s in store[level]], axis=0)
    far_all = np.mean([s[1] for s in store[level]], axis=0)
    far_unk = np.mean([s[2] for s in store[level]], axis=0)
    e, t = eer(frr, far_all)

    print(f"\n[{level} level] threshold -> FRR (real user rejected) | FAR all | FAR strangers")
    for th in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
        i = int(round(th * 100))
        print(f"  thr {th:.1f}: FRR {frr[i]*100:5.1f}% | FAR {far_all[i]*100:5.1f}% | FAR strangers {far_unk[i]*100:5.1f}%")
    print(f"  EER point: threshold {t:.2f}, error {e*100:.1f}%")

    ax.plot(THR, frr * 100, label="FRR (real user rejected)")
    ax.plot(THR, far_all * 100, label="FAR (any impostor accepted)")
    ax.plot(THR, far_unk * 100, "--", label="FAR (strangers only)")
    ax.axvline(t, color="gray", ls=":", label=f"EER {e*100:.1f}% @ {t:.2f}")
    ax.set_title(f"Authentication, {level} level")
    ax.set_xlabel("Acceptance threshold")
    ax.grid(alpha=0.3)
    for i, th in enumerate(THR):
        curve_rows.append({"level": level, "threshold": th, "FRR": frr[i],
                           "FAR_all": far_all[i], "FAR_strangers": far_unk[i]})
axes[0].set_ylabel("Error rate (%)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig("auth_far_frr.png", dpi=150)
plt.close(fig)
pd.DataFrame(curve_rows).to_csv("auth_curves.csv", index=False)

# ---- model for the app (split 1): enrolled people only + suggested thresholds
first_model["threshold_window"] = float(first_split["window"])
first_model["threshold_recording"] = float(first_split["recording"])
joblib.dump(first_model, "brainprint_auth_rf.joblib")
print("\nSaved: brainprint_auth_rf.joblib, auth_far_frr.png, auth_curves.csv")

=== PART 1: identification, 30 subjects ===
Window level    :  98.1%  (1814 windows, never seen in training)
Recording level :  98.3%  (60 recordings, probabilities averaged)
Random guess    :   3.3%
Saved: brainprint_rf_30.joblib, confusion_matrix_30.png

=== PART 2: authentication, 20 enrolled / 10 strangers, 5 random splits ===
split 1/5 done | EER window 2.0% | EER recording 2.1%
split 2/5 done | EER window 2.7% | EER recording 2.5%
split 3/5 done | EER window 1.9% | EER recording 1.9%
split 4/5 done | EER window 2.3% | EER recording 2.2%
split 5/5 done | EER window 1.7% | EER recording 0.5%

Equal Error Rate (EER, lower = better), mean +- std over splits:
  window   :   2.1% +- 0.4
  recording:   1.8% +- 0.7

[window level] threshold -> FRR (real user rejected) | FAR all | FAR strangers
  thr 0.1: FRR   0.3% | FAR   7.2% | FAR strangers  15.6%
  thr 0.2: FRR   1.7% | FAR   2.5% | FAR strangers   6.2%
  thr 0.3: FRR   4.0% | FAR   1.1% | FAR strangers   3.1%
  thr 0.4: FRR   7.8% |

In [6]:
%pip install streamlit joblib matplotlib mne numpy pandas scipy scikit-learn

  Using cached streamlit-1.67.0-py3-none-any.whl.metadata (10 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached mne-1.13.2-py3-none-any.whl.metadata (17 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached starlette-1.10.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached python_multipart-0.0.33-py3-none-any.whl.metadata (6.3 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-7.1.0-py3-none-win_amd64.whl.metadata (62 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached lazy_loader-0.6-py3-none-any.whl.metadata (6.0 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
  U


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
